# Week 13-1 · EFS-05 — Trading ETFs

**An ETF deep-dive — no programming prerequisites, no markets experience assumed.** The
instructor (25 years in markets, a decade at Goldman Sachs running an ETF book) builds the whole
picture from scratch and *from practice*, not theory. This notebook reproduces every
**Excel/pencil exercise** he runs in the live class so the mechanics stick.

**The arc of the session:**
1. **Cap-weighted indices** — what an index *is*, and the **divisor** that keeps it stable through corporate actions.
2. **Fund structures** — open-/closed-ended funds, NAV, and the **two fatal flaws of mutual funds**.
3. **The primary market** — the ETF as a *warehouse*: **creation & redemption baskets**, in-kind, authorized participants.
4. **How ETFs fix the two flaws** — the free-rider (liquidity) problem and the tax problem, both solved by **in-kind** transactions.
5. **The secondary market** — **iNAV**, premium/discount, and the **arbitrage** that keeps price ≈ NAV (where HFTs live).

> Everything below is computed in Python from small baskets that mirror the lecture's Indian-market
> example (BEL, Coal India, Concor…). Numbers are illustrative but internally consistent.

In [1]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print("ready")

ready


## 1. Cap-weighted indices and the divisor

A **cap-weighted index** weights each stock by its **market value = price × shares**. Almost every
famous index (Sensex, NIFTY, S&P 500, NASDAQ) is cap-weighted. An **index provider** (MSCI, FTSE
Russell, S&P, or India's IISL) doesn't manage money — it just *builds and maintains* indices, and
charges everyone who tracks them. These providers are now hugely powerful: if MSCI drops a country
from an index, billions of tracking dollars follow.

The trick that keeps an index a clean, comparable number is the **divisor**:

$$\text{Index} = \frac{\text{Total market value}}{\text{Divisor}}$$

At launch you *normalize* a messy total (say ₹7.5 m) to a round number (say 1000) by choosing the
divisor. Then, whenever a **corporate action** changes shares/constituents but **not** real value,
you **re-solve the divisor** so the index doesn't jump.

In [2]:
# The lecture's 2-stock index at t-1
idx = pd.DataFrame({"price":[100.0, 25.0], "shares":[50_000, 100_000]}, index=["X1","X2"])
idx["mkt_value"] = idx.price * idx.shares
total_mv = idx.mkt_value.sum()
INDEX_START = 1000.0
divisor = total_mv / INDEX_START
print(idx)
print(f"\nTotal MV = {total_mv:,.0f}   Index = {INDEX_START:.0f}   => Divisor = {divisor:,.1f}")

      price  shares      mkt_value
X1 100.0000   50000 5,000,000.0000
X2  25.0000  100000 2,500,000.0000

Total MV = 7,500,000   Index = 1000   => Divisor = 7,500.0


In [3]:
def index_from(df, divisor):
    return df.eval("price*shares").sum() / divisor

def resolve_divisor(df, target_index):
    return df.eval("price*shares").sum() / target_index

# --- Corporate action A: 2-for-1 SPLIT on X1 (price halves, shares double) ---
a = idx[["price","shares"]].copy()
a.loc["X1","price"]  /= 2
a.loc["X1","shares"] *= 2
mv_a = a.eval("price*shares").sum()
print("A) 2-for-1 split on X1")
print(f"   Total MV: {total_mv:,.0f} -> {mv_a:,.0f}  (UNCHANGED) => index & divisor UNCHANGED")
print(f"   index stays {index_from(a, divisor):.1f}, divisor stays {divisor:,.1f}\n")

# --- Corporate action B: $2 DIVIDEND on X1 (price drops by 2, MV falls) ---
b = idx[["price","shares"]].copy()
b.loc["X1","price"] -= 2.0
mv_b = b.eval("price*shares").sum()
div_b = resolve_divisor(b, INDEX_START)          # keep index at 1000 -> re-solve divisor
print("B) $2 dividend on X1")
print(f"   Total MV: {total_mv:,.0f} -> {mv_b:,.0f}  (FELL) ; keep index 1000")
print(f"   => divisor changes {divisor:,.1f} -> {div_b:,.1f}\n")

# --- Corporate action C: CONSTITUENT change (X2 out, X3 in) ---
c = pd.DataFrame({"price":[100.0, 50.0], "shares":[50_000, 200_000]}, index=["X1","X3"])
mv_c = c.eval("price*shares").sum()
div_c = resolve_divisor(c, INDEX_START)
print("C) X2 leaves, X3 enters (price 50, shares 200,000)")
print(f"   Total MV: {total_mv:,.0f} -> {mv_c:,.0f}  (JUMPED) ; keep index 1000")
print(f"   => divisor changes {divisor:,.1f} -> {div_c:,.1f}")

A) 2-for-1 split on X1
   Total MV: 7,500,000 -> 7,500,000  (UNCHANGED) => index & divisor UNCHANGED
   index stays 1000.0, divisor stays 7,500.0

B) $2 dividend on X1
   Total MV: 7,500,000 -> 7,400,000  (FELL) ; keep index 1000
   => divisor changes 7,500.0 -> 7,400.0

C) X2 leaves, X3 enters (price 50, shares 200,000)
   Total MV: 7,500,000 -> 15,000,000  (JUMPED) ; keep index 1000
   => divisor changes 7,500.0 -> 15,000.0


**The rule the exercises teach:** a corporate action that **doesn't change real value** (a split)
leaves both index *and* divisor untouched; one that **changes total MV** without being a real
return (a dividend, or swapping one constituent for another) keeps the **index** continuous by
**re-solving the divisor**. Divisors are recomputed **once a day** (corporate actions are announced
well in advance), not in real time — even though on a 500-stock index *something* happens almost
every day.

## 2. Fund structures, NAV, and the two flaws of mutual funds

**Collective investment vehicles** pool many investors' money. Key split:
- **Open-ended** (most mutual funds): units are **created/redeemed on demand**, so the unit count
  floats; usually **no secondary market**. It's an **AUM game** — bigger AUM, more marketing muscle.
- **Closed-ended**: a fixed pot raised once, then **trades on a secondary market**. Its market price
  can drift far from **NAV** (= total fund value ÷ units) because there's **no arbitrage mechanism**
  back to the fund.

$$\text{NAV} = \frac{\text{Total portfolio value}}{\text{Number of units}}$$

Modern mutual funds price at the **end-of-day NAV** ("NAV-next"), which sounds fair but has spawned
decades of scandals (late trading, market timing). Two structural flaws matter for *why ETFs exist*.

In [4]:
# --- FLAW 1: the "free-rider" / shared-liquidity problem ---
# 3 investors put in 1,000,000 each. On day 2 one redeems; fund must SELL, paying a txn cost.
investors = 3
contribution = 1_000_000
total_assets = investors * contribution
redeem_txn_cost = 2_000.0          # cost to raise cash for the redeemer

# The 2,000 cost is charged at FUND level -> socialized across ALL current investors
naive_expectation = contribution - redeem_txn_cost   # 998,000 if redeemer bore full cost
cost_per_investor = redeem_txn_cost / investors
redeemer_actually_gets = contribution - cost_per_investor
print(f"Redeemer naively expects (bears full cost): {naive_expectation:,.0f}")
print(f"But cost is socialized: {redeem_txn_cost:,.0f} / {investors} = {cost_per_investor:,.0f} each")
print(f"=> Redeemer actually gets ~{redeemer_actually_gets:,.0f}  (GREATER than {naive_expectation:,.0f})")
print("\nThe trader who pops in/out is FREE-RIDING; long-term investors subsidize the txn costs.")
print("Studies suggest this can cost long-term holders ~1.5-2% CAGR over time.")

Redeemer naively expects (bears full cost): 998,000
But cost is socialized: 2,000 / 3 = 667 each
=> Redeemer actually gets ~999,333  (GREATER than 998,000)

The trader who pops in/out is FREE-RIDING; long-term investors subsidize the txn costs.
Studies suggest this can cost long-term holders ~1.5-2% CAGR over time.


In [5]:
# --- FLAW 2: the tax problem for an active fund ---
# An active manager who SELLS a winner triggers a taxable capital gain DISTRIBUTED to all holders,
# regardless of each holder's own tax situation.
gain_realized = 500_000.0
cap_gains_tax_rate = 0.15
tax_distributed = gain_realized * cap_gains_tax_rate
print(f"Active manager sells a winner: realized gain {gain_realized:,.0f}")
print(f"Tax event of {tax_distributed:,.0f} is forced onto ALL investors, even long-term holders who didn't sell.")
print("So every sell decision drags in tax consequences for the whole book.")

Active manager sells a winner: realized gain 500,000
Tax event of 75,000 is forced onto ALL investors, even long-term holders who didn't sell.
So every sell decision drags in tax consequences for the whole book.


**The history:** in 1976 (≈45 years ago) Prof. **Nils Hakansson**'s paper described these flaws
verbally. A decade later **Nathan Most** — a *commodity* trader — read it and built a product to fix
them: **SPDR ("Spiders")**, the first ETF, launched **1993**. Most's commodity background is why ETF
jargon (warehouses, baskets, receipts) smells like commodity markets.

## 3. The primary market — the ETF as a warehouse

Picture the ETF fund manager as a **warehouse manager**, not a stock-picker. You can't buy *one*
ETF unit from the fund; you transact in **lots** (a **creation unit**, e.g. 100,000 shares). And the
transaction is **in kind** — **no cash changes hands for the stocks**:

- **Creation:** an **Authorized Participant (AP)** *assembles the creation basket* (the exact stocks
  & weights the fund publishes daily) **+ a small cash component**, hands it to the fund, and
  receives ETF shares.
- **Redemption:** the AP returns ETF shares and receives a **basket of stocks** back (then sells them
  itself if it wants cash).

APs are **special** (≈30+ for SPDR originally; today often HFT firms) — big balance sheets, the only
ones allowed in the primary market.

In [6]:
# A 10-stock creation basket (illustrative, BEL/Coal India/Concor flavour). Per CREATION UNIT.
basket = pd.DataFrame({
    "ticker": ["BEL","COALINDIA","CONCOR","INFY","TCS","HDFCBANK","RELIANCE","ITC","SBIN","LT"],
    "qty":    [   33,      1686,     123,   210,    95,      260,       180,  640,  410,  150],
    "price":  [288.5,     402.1,   745.0,1520.0,3850.0,   1660.0,    2480.0,445.0,820.0,3580.0],
})
basket["value"] = basket.qty * basket.price
stock_value = basket.value.sum()
CASH_COMPONENT = 45_325.0           # fund wants some cash (for rebalances / corp actions)
TXN_BPS = 20 / 10_000               # 20 bps transaction charge
ETF_SHARES = 100_000                # one creation unit

print(basket.to_string(index=False))
print(f"\nStock value of basket: {stock_value:,.2f}")

   ticker  qty      price        value
      BEL   33   288.5000   9,520.5000
COALINDIA 1686   402.1000 677,940.6000
   CONCOR  123   745.0000  91,635.0000
     INFY  210 1,520.0000 319,200.0000
      TCS   95 3,850.0000 365,750.0000
 HDFCBANK  260 1,660.0000 431,600.0000
 RELIANCE  180 2,480.0000 446,400.0000
      ITC  640   445.0000 284,800.0000
     SBIN  410   820.0000 336,200.0000
       LT  150 3,580.0000 537,000.0000

Stock value of basket: 3,500,046.10


In [7]:
# CREATION: cost to the AP to obtain 100,000 ETF shares
buy_txn = stock_value * TXN_BPS
creation_cost = stock_value + buy_txn + CASH_COMPONENT
cost_per_etf = creation_cost / ETF_SHARES
print("--- CREATION (assemble basket -> get ETF shares) ---")
print(f"  stock value           : {stock_value:,.2f}")
print(f"  + 20 bps txn on buys   : {buy_txn:,.2f}")
print(f"  + cash component       : {CASH_COMPONENT:,.2f}")
print(f"  = total creation cost  : {creation_cost:,.2f}")
print(f"  / {ETF_SHARES:,} ETF shares  => COST per ETF = {cost_per_etf:,.4f}")

--- CREATION (assemble basket -> get ETF shares) ---
  stock value           : 3,500,046.10
  + 20 bps txn on buys   : 7,000.09
  + cash component       : 45,325.00
  = total creation cost  : 3,552,371.19
  / 100,000 ETF shares  => COST per ETF = 35.5237


In [8]:
# REDEMPTION: hand back 100,000 ETF shares -> receive basket -> SELL it
sell_txn = stock_value * TXN_BPS
redemption_proceeds = stock_value - sell_txn + CASH_COMPONENT
value_per_etf = redemption_proceeds / ETF_SHARES
print("--- REDEMPTION (give ETF shares -> get basket -> sell) ---")
print(f"  stock sale value       : {stock_value:,.2f}")
print(f"  - 20 bps txn on sells  : {sell_txn:,.2f}")
print(f"  + cash component       : {CASH_COMPONENT:,.2f}")
print(f"  = total proceeds       : {redemption_proceeds:,.2f}")
print(f"  / {ETF_SHARES:,} ETF shares  => SELL value per ETF = {value_per_etf:,.4f}")
print(f"\nRound-trip cost (the AP's burden): {cost_per_etf - value_per_etf:,.4f} per ETF "
      f"= 2 x 20bps = {2*TXN_BPS*stock_value/ETF_SHARES:,.4f}")

--- REDEMPTION (give ETF shares -> get basket -> sell) ---
  stock sale value       : 3,500,046.10
  - 20 bps txn on sells  : 7,000.09
  + cash component       : 45,325.00
  = total proceeds       : 3,538,371.01
  / 100,000 ETF shares  => SELL value per ETF = 35.3837

Round-trip cost (the AP's burden): 0.1400 per ETF = 2 x 20bps = 0.1400


## 4. How in-kind baskets solve BOTH flaws

Replay the 3-investor day-1/day-2 story, but as an ETF. Investors bring **baskets** (not cash);
the manager just hands back ETF shares. When one redeems, he hands back a **basket** and
extinguishes the shares — the **NAV for everyone else is untouched**.

In [9]:
# 3 APs each bring a basket worth 2.5m -> get 100,000 ETF shares each
basket_val = 2_500_000.0
units_each = 100_000
day1_units = 3 * units_each
day1_assets = 3 * basket_val
nav_day1 = day1_assets / day1_units
print(f"Day 1: 3 baskets, assets {day1_assets:,.0f}, units {day1_units:,} -> NAV = {nav_day1:.2f}")

# Day 2: one redeems -> hand back a basket, extinguish its 100,000 units
day2_units = day1_units - units_each
day2_assets = day1_assets - basket_val
nav_day2 = day2_assets / day2_units
print(f"Day 2: 1 redeems in-kind, assets {day2_assets:,.0f}, units {day2_units:,} -> NAV = {nav_day2:.2f}")
print(f"\nNAV unchanged ({nav_day1:.2f} -> {nav_day2:.2f}): remaining holders bear NO txn cost.")
print("FLAW 1 solved: the AP (not other investors) carries the transaction burden.")
print("FLAW 2 solved: in-kind delivery realizes NO capital gain -> no tax forced on holders.")

Day 1: 3 baskets, assets 7,500,000, units 300,000 -> NAV = 25.00
Day 2: 1 redeems in-kind, assets 5,000,000, units 200,000 -> NAV = 25.00

NAV unchanged (25.00 -> 25.00): remaining holders bear NO txn cost.
FLAW 1 solved: the AP (not other investors) carries the transaction burden.
FLAW 2 solved: in-kind delivery realizes NO capital gain -> no tax forced on holders.


**Baskets must track the index.** Unlike a static warehouse, the basket's constituents **change
when the index rebalances** (a stock enters/leaves). That's partly why the fund collects the
**cash component** — to fund rebalances and corporate actions, since it never takes cash for the
stocks. Index changes are rare (NIFTY 50 ≈ 4 changes in ~15 years; NIFTY Next-50 ≈ 8), so an ETF
manager **barely trades** day to day — the opposite of an active manager.

## 5. The secondary market — iNAV, premium/discount, and arbitrage

ETFs trade all day on a **lit order book** (everyone sees best bid/ask). Throughout the day an
**indicative NAV (iNAV)** is published — just the live basket value ÷ units. APs uniquely see
**both** markets, so they **arbitrage** the gap and keep price ≈ NAV:

- **Premium:** secondary bid/ask **above** iNAV → the ETF is "rich" → **sell** in the secondary
  (or post a limit sell just below the ask).
- **Discount:** secondary **below** iNAV → the ETF is "cheap" → **buy**.

This linkage (absent in closed-ended funds) is exactly why an ETF's price hugs its NAV.

In [10]:
def inav_from_basket(basket_df, cash, units):
    return (basket_df.value.sum() + cash) / units

iNAV = inav_from_basket(basket, CASH_COMPONENT, ETF_SHARES)
print(f"iNAV (fair value of 1 ETF) = {iNAV:,.4f}\n")

def arb_decision(best_bid, best_ask, inav, markup_bps=0.0):
    m = markup_bps / 10_000
    # Premium: market above fair value by more than required markup -> SELL
    if best_bid > inav * (1 + m):
        return f"PREMIUM: market {best_bid:.2f}/{best_ask:.2f} > iNAV {inav:.2f} -> SELL (limit just below ask)"
    # Discount: market below fair value by more than required markup -> BUY
    if best_ask < inav * (1 - m):
        return f"DISCOUNT: market {best_bid:.2f}/{best_ask:.2f} < iNAV {inav:.2f} -> BUY (limit just above bid)"
    return f"WITHIN {markup_bps:.0f}bps band of iNAV {inav:.2f} -> NO trade (edge too thin)"

# Case 1: any edge is enough
print("No-markup arbitrage:")
print("  ", arb_decision(best_bid=iNAV+0.08, best_ask=iNAV+0.13, inav=iNAV))
print("  ", arb_decision(best_bid=iNAV-0.13, best_ask=iNAV-0.08, inav=iNAV))
# Case 2: require a 35 bps markup to bother
print("\nRequire 35 bps markup:")
print("  ", arb_decision(best_bid=iNAV+0.05, best_ask=iNAV+0.09, inav=iNAV, markup_bps=35))
print("  ", arb_decision(best_bid=iNAV+0.20, best_ask=iNAV+0.25, inav=iNAV, markup_bps=35))

iNAV (fair value of 1 ETF) = 35.4537

No-markup arbitrage:
   PREMIUM: market 35.53/35.58 > iNAV 35.45 -> SELL (limit just below ask)
   DISCOUNT: market 35.32/35.37 < iNAV 35.45 -> BUY (limit just above bid)

Require 35 bps markup:
   WITHIN 35bps band of iNAV 35.45 -> NO trade (edge too thin)
   PREMIUM: market 35.65/35.70 > iNAV 35.45 -> SELL (limit just below ask)


## 6. Summary chart — the divisor through corporate actions, and the arbitrage band

In [11]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
IND="#4f46e5"; IND2="#818cf8"; GRN="#16a34a"; RED="#dc2626"; GY="#9ca3af"

# (1) Divisor across corporate actions
labels = ["t-1\nbase", "Split\n(2:1)", "Dividend\n($2)", "Constituent\n(X2->X3)"]
divisors = [divisor, divisor, div_b, div_c]
indices  = [INDEX_START]*4
bars = ax[0].bar(labels, divisors, color=[GY, IND2, IND, IND], edgecolor="black", lw=0.6)
ax[0].set_title("Divisor re-solved to keep index = 1000", fontweight="bold")
ax[0].set_ylabel("divisor")
for b, d in zip(bars, divisors):
    ax[0].text(b.get_x()+b.get_width()/2, d, f"{d:,.0f}", ha="center", va="bottom", fontsize=9)
ax[0].text(0.02, 0.95, "index stays 1000 throughout", transform=ax[0].transAxes,
           color=GRN, fontsize=9, va="top")

# (2) Premium/discount arbitrage band around iNAV
m = 35/10_000
x = np.arange(5)
mkt = np.array([iNAV-0.30, iNAV-0.05, iNAV, iNAV+0.05, iNAV+0.30])
ax[1].axhspan(iNAV*(1-m), iNAV*(1+m), color=IND2, alpha=0.25, label="35 bps no-trade band")
ax[1].axhline(iNAV, color=IND, lw=1.5, label=f"iNAV {iNAV:.2f}")
colors = [GRN if v < iNAV*(1-m) else RED if v > iNAV*(1+m) else GY for v in mkt]
ax[1].scatter(x, mkt, c=colors, s=90, zorder=5, edgecolor="black", lw=0.5)
for xi, v, c in zip(x, mkt, colors):
    tag = "BUY" if c==GRN else "SELL" if c==RED else "—"
    ax[1].text(xi, v+0.02, tag, ha="center", fontsize=8, color=c, fontweight="bold")
ax[1].set_title("Secondary price vs iNAV -> arbitrage", fontweight="bold")
ax[1].set_ylabel("ETF price"); ax[1].set_xticks([]); ax[1].legend(fontsize=8, loc="lower right")

plt.tight_layout(); plt.savefig("chart_1_etf.png", dpi=115, bbox_inches="tight")
print("saved chart_1_etf.png")

saved chart_1_etf.png


## 7. The one-paragraph version

An **ETF** tracks a **cap-weighted index** (value = price × shares), kept stable through corporate
actions by **re-solving the divisor** so the index never jumps (splits change nothing; dividends and
constituent swaps change the divisor, recomputed once a day). ETFs exist to fix **two flaws of
mutual funds**: the **free-rider/liquidity problem** (when a trader pops in or out, *other* holders
foot the transaction cost — a ~1.5–2% CAGR drag over time) and the **tax problem** (an active
manager's sell forces capital-gains tax on everyone). The fix is the **primary market**, where the
ETF acts like a **warehouse**: **authorized participants** assemble a published **creation basket**
(+ a small **cash component**) and swap it **in kind** for ETF shares, or redeem shares back into a
basket — **no cash for the stocks**, so the AP (not other holders) bears costs and **no capital
gain is realized**. In the **secondary market**, an **iNAV** is published live, and APs — uniquely
seeing both markets — **arbitrage** premiums (sell) and discounts (buy), keeping the ETF's price
glued to its NAV. That arbitrage, impossible in closed-ended funds, is where HFTs and market makers
make their money.

# Additive Workbook Validation Section

The following cells are appended from `Week 13-1 EFS-05 Trading ETFs_resource_addendum_practice.ipynb`. They preserve the original illustrative calculations and add the official workbook-backed version.

## 1. Load workbook-derived references

The source workbook has six slide sheets. Slide-21 gives quantities and the cash component; Slide-22 gives the base prices used for corrected creation/redemption economics.

In [12]:
from pathlib import Path
import pandas as pd
import numpy as np

code_dir = Path.cwd()
sheets = pd.read_csv(code_dir / "efs05_workbook_sheet_inventory.csv")
basket = pd.read_csv(code_dir / "efs05_source_creation_basket.csv")
corrected = pd.read_csv(code_dir / "efs05_workbook_creation_redemption_corrected.csv")
print(sheets.to_string(index=False))
print("\nSource basket value:", f"{basket['Value'].sum():,.2f}")

   sheet  rows  columns                                                role
Slide-21    12        3       creation basket quantities and cash component
Slide-22    11        4             basket quantities with reference prices
Slide-24    11        4             basket quantities with reference prices
Slide-33    11        4             basket quantities with reference prices
Slide-39    11        3 secondary-market price snapshot for the same basket
Slide-44    11        3 secondary-market price snapshot for the same basket

Source basket value: 2,434,333.90


## 2. Correct creation and redemption economics from the workbook

The existing local notebook uses a separate illustrative 3.50m basket. The workbook-backed correction uses the actual BEL/COALINDIA/CONCOR basket shipped with EFS-05.

In [13]:
comparison = pd.read_csv(code_dir / "efs05_existing_vs_workbook_basket_comparison.csv")
print(corrected.to_string(index=False))
print("\nComparison with preserved illustration:")
print(comparison.to_string(index=False))

                     item  creation_rupees  redemption_rupees
       stock basket value   2,434,333.9000     2,434,333.9000
20 bps transaction charge       4,868.6678        -4,868.6678
           cash component      45,325.0000        45,325.0000
                    total   2,484,527.5678     2,474,790.2322
            per ETF share          24.8453            24.7479

Comparison with preserved illustration:
                                   source    stock_value  cash_component  creation_per_etf  redemption_per_etf                              status  round_trip_cost_per_etf
existing local illustrative notebook/page 3,500,046.1000     45,325.0000           35.5237             35.3837             preserved; illustrative                   0.1400
               official workbook Slide-22 2,434,333.9000     45,325.0000           24.8453             24.7479 added as workbook-backed correction                   0.0974


## 3. Secondary-market snapshots

Slide-39 and Slide-44 provide later price snapshots for the same basket. Reusing Slide-21 quantities gives the live basket value, NAV per ETF, and creation/redemption bands.

In [14]:
secondary = pd.read_csv(code_dir / "efs05_secondary_market_snapshots.csv")
print(secondary.to_string(index=False))

   sheet    stock_value  cash_component  nav_per_etf  creation_per_etf  redemption_per_etf
Slide-22 2,434,333.9000     45,325.0000      24.7966           24.8453             24.7479
Slide-39 2,444,588.9900     45,325.0000      24.8991           24.9480             24.8502
Slide-44 2,468,156.6600     45,325.0000      25.1348           25.1842             25.0855


## 4. Index divisor and strategy-model references

The transcript extends beyond the workbook: it covers divisor continuity, low-frequency factor ETF models, mid-frequency allocation, high-frequency market making, active ETFs, and leveraged ETFs.

In [15]:
divisor = pd.read_csv(code_dir / "efs05_index_divisor_reference.csv")
models = pd.read_csv(code_dir / "efs05_trading_model_reference.csv")
print(divisor.to_string(index=False))
print("\nETF strategy ladder:")
print(models[["frequency", "model"]].to_string(index=False))

           event  total_market_value  index_level  divisor                                                                       correct_rule
            base             7500000         1000     7500             initialize divisor as total market value divided by chosen index level
   2-for-1 split             7500000         1000     7500    split changes price and shares but not market value, so divisor stays unchanged
2 dividend on X1             7400000         1000     7400 cash dividend lowers price/market value; re-solve divisor to keep index continuous
constituent swap            15000000         1000    15000                   new constituent changes aggregate market value; re-solve divisor

ETF strategy ladder:
        frequency                                model
              low        factor ETF or factor exposure
              mid ETF asset allocation / robo-advisory
             high                    ETF market making
product structure                    active ETF 